In [ ]:
import requests
import json
import pandas as pd
import time
import folium

from datetime import datetime

In [ ]:
# PEYTON RUN
filepath = "/Users/peyton/Desktop/TicketmasterAPIKey.txt"

In [ ]:
# KATRIEL RUN
filepath = "/Users/katriellayam/Downloads/ticketmaster_project.txt"

In [ ]:
# LEAH RUN
filepath = "/Users/leahaviles/Downloads/ticket_masterAPIKey.txt"

In [ ]:
# JOY RUN
filepath = "/Users/joy/Downloads/ticketmaster api key.txt"

In [ ]:
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

key = read_key(filepath)

type(key)

In [ ]:
url = f"https://app.ticketmaster.com/discovery/v2/venues.json?apikey={key}"

response = requests.get(url)

response.raise_for_status()

test_data = response.json()

test_data

In [ ]:
venues = test_data['_embedded']['venues']

df = pd.DataFrame([{
    'id': v.get('id'),
    'name': v.get('name'),
    'address': v.get('address', {}).get('line1'),
    'city': v.get('city', {}).get('name'),
    'state': v.get('state', {}).get('stateCode'),
    'latitude': v.get('location', {}).get('latitude'),
    'longitude': v.get('location', {}).get('longitude'),
    'postal_code': v.get('postalCode'),
    'upcoming_events_total': v.get('upcomingEvents', {}).get('_total'),
} for v in venues])

df = df.sort_values(by='upcoming_events_total', ascending=False) # order df in terms of most upcoming events
df = df.drop(columns='id') # remove id column

df

In [ ]:
m = folium.Map(location=[38, -95], zoom_start=4)

for _, row in df.iterrows():
    if row["latitude"] is None or row["longitude"] is None:
        continue

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['address']}<br>"
        f"{row['city']}, {row['state']} {row['postal_code']}<br>"
        f"Upcoming Events: {row['upcoming_events_total']}"
    )

    folium.Marker(
        location=[float(row["latitude"]), float(row["longitude"])],
        popup=folium.Popup(popup_text, max_width=200),
        icon=folium.Icon(color="darkblue", icon="ticket", prefix="fa")
    ).add_to(m)

m.save("venues.html")
m

In [ ]:
# fetch events per month
import time

all_events = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while True:
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()

        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})
            all_events.append({
                "name": e.get("name"),
                "date": e.get("dates", {}).get("start", {}).get("localDate"),
                "venue": venue.get("name"),
                "city": venue.get("city", {}).get("name"),
                "state": venue.get("state", {}).get("stateCode"),
                "latitude": loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "url": e.get("url"),
                "month": start[:7],
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events)} events so far")

        if page + 1 >= total_pages:
            break

        page += 1
        time.sleep(0.25)  # rate limit buffer

events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"])
events_df["latitude"] = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)

print(f"\nTotal events: {len(events_df)}")

In [ ]:
import sqlite3

# create in-memory db and load df
conn = sqlite3.connect(":memory:")
events_df.to_sql("events", conn, index=False, if_exists="replace")
df.to_sql("venues", conn, index=False, if_exists="replace")

# events by segment
pd.read_sql("""
    SELECT segment, COUNT(*) as count
    FROM events
    GROUP BY segment
    ORDER BY count DESC
""", conn)

In [ ]:
# only include events in mainland US
events_df = pd.read_sql("""
    SELECT *
    FROM events
    WHERE latitude BETWEEN 24.5 AND 49.5
    AND longitude BETWEEN -125.0 AND -66.5
""", conn)

print(f"Events before: {len(events_df)}")

# update the sql table
events_df.to_sql("events", conn, index=False, if_exists="replace")

print(f"Events after: {len(events_df)}")

events_df

In [ ]:
# only include events in US
# categorize events by types of color

m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

# color by segment
segment_colors = {
    "Music": "blue",
    "Sports": "red",
    "Arts & Theatre": "green",
    "Miscellaneous": "gray",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in events_df.iterrows():
    month = row["month"]
    if month not in groups:
        continue

    color = segment_colors.get(row["segment"], "gray")

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}<br>"
        f"Segment: {row['segment']}<br>"
        f"Genre: {row['genre']}"
    )

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [ ]:
# can toggle event locations per month
m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in events_df.iterrows():
    month = row["month"]
    if month not in groups:
        continue
    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="darkblue",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [ ]:
# this time we fetch 'classifications' gives if its music, sport, etc.
all_events = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while True:
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()

        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})

            # extract classifications
            classifications = e.get("classifications", [{}])
            primary = classifications[0] if classifications else {}
            segment  = primary.get("segment", {}).get("name")
            genre    = primary.get("genre", {}).get("name")
            subgenre = primary.get("subGenre", {}).get("name")

            all_events.append({
                "name":     e.get("name"),
                "date":     e.get("dates", {}).get("start", {}).get("localDate"),
                "venue":    venue.get("name"),
                "city":     venue.get("city", {}).get("name"),
                "state":    venue.get("state", {}).get("stateCode"),
                "latitude":  loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "url":      e.get("url"),
                "month":    start[:7],
                "segment":  segment,
                "genre":    genre,
                "subgenre": subgenre,
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events)} events so far")

        if page + 1 >= total_pages:
            break

        page += 1
        time.sleep(0.25)

events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"])
events_df["latitude"]  = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)

print(f"\nTotal events: {len(events_df)}")
print(events_df[["segment", "genre", "subgenre"]].value_counts().head(20))

In [ ]:
# plot categories
import matplotlib.pyplot as plt

genre_counts = (
    events_df[~events_df["genre"].isin(["Undefined", "Miscellaneous", "Other"])]
    .groupby(["segment", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# color by segment
segment_colors = {
    "Music": "steelblue",
    "Sports": "firebrick",
    "Arts & Theatre": "mediumseagreen",
    "Miscellaneous": "gray",
}
colors = genre_counts["segment"].map(segment_colors).fillna("gray")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    genre_counts["genre"] + " (" + genre_counts["segment"] + ")",
    genre_counts["count"],
    color=colors
)

ax.set_xlabel("Number of Events")
ax.set_title("Top 20 Event Genres on Ticketmaster (2026)", fontsize=14)
ax.invert_yaxis()

# legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in segment_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()